# Filtro 1

## Entreno de modelo

### Importación de librerías

In [ ]:
import os
import numpy as np
import joblib
from deepface import DeepFace
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from tqdm import tqdm
import kagglehub

### Importación de dataset y definición de categorías

In [2]:
DATASET_ID  = "saramhai/people-with-and-without-glasses-dataset"
DOWNLOAD_PATH  = kagglehub.dataset_download(DATASET_ID)

CLASS_LABELS  = ["glasses", "no_glasses"] 
FACE_MODEL = "ArcFace"
MODEL_FILENAME = "glasses_classifier.pkl"
LABELS_FILE = "glasses_labels.pkl"

embeddings = []
labels = []


### Entrenamiento del dataset

In [ ]:
base_path = os.path.join(DOWNLOAD_PATH , "images")

for label_id, label_name in enumerate(CLASS_LABELS):
    label_path = os.path.join(base_path, label_name)
    if not os.path.exists(label_path):
        print(f"¡Aviso! No se encontró la carpeta: {label_path}")
        continue

    img_files = [f for f in os.listdir(label_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Mezclar aleatoriamente
    np.random.shuffle(img_files)

    print(f"\nProcesando categoría: {label_name} (Usando {len(img_files)} imágenes)")

    # Extraer embeddings
    for filename in tqdm(selected_files, desc=label_name):
        file_path = os.path.join(label_path, filename)
        try:
            rep_data = DeepFace.represent(
                img_path=file_path,
                model_name=FACE_MODEL,
                enforce_detection=False,
                detector_backend='skip'
            )

            face_vector = rep_data[0]["embedding"]
            embeddings.append(face_vector)
            labels.append(label_id)

        except Exception as err:
            if 'face could not be detected' not in str(err):
                print(f"Error procesando {file_path}: {err}")

if not embeddings:
    print("Error: No se extrajo ningún embedding. Verifica el dataset.")
    exit()

embeddings = np.array(embeddings)
labels = np.array(labels)

print(f"\nTotal de embeddings extraídos: {embeddings.shape[0]}")
print(f"Dimensiones de cada embedding: {embeddings.shape[1]}")
print(f"Distribución de clases: {np.bincount(labels)}")

# Evaluación con train/test split
print("\nEvaluando modelo con split 70/30...")
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.3, random_state=42, stratify=labels
)

temp_model = SVC(kernel='linear', probability=True, random_state=42)
temp_model.fit(X_train, y_train)

y_pred = temp_model.predict(X_test)
print("\n" + classification_report(y_test, y_pred, target_names=CLASS_LABELS))

# Entrenamiento final con todos los datos
print("\nEntrenando modelo final con todos los datos...")
best_model = SVC(kernel='linear', probability=True, random_state=42)
best_model.fit(embeddings, labels)

# Guardar modelo y categorías
joblib.dump(best_model, MODEL_FILENAME)
joblib.dump(CLASS_LABELS, LABELS_FILE)

print(f"\nModelo: '{MODEL_FILENAME}'")
print(f"Categorías: '{LABELS_FILE}'")

### Uso

In [ ]:
import cv2
import numpy as np
import joblib
from deepface import DeepFace

# --- Configuración ---
MODEL_FILENAME = "glasses_classifier.pkl"
CATEGORIES_FILENAME = "glasses_labels.pkl"
FACE_MODEL = "ArcFace"  # ¡Debe ser el mismo que en el entrenamiento!

# --- Cargar modelos ---
try:
    model = joblib.load(MODEL_FILENAME)
    categories = joblib.load(CATEGORIES_FILENAME)
except FileNotFoundError:
    print(f"Error: No se encontraron los archivos '{MODEL_FILENAME}' o '{CATEGORIES_FILENAME}'.")
    print("Asegúrate de ejecutar el script de entrenamiento primero.")
    exit()

# Detector de rostros de OpenCV (para encontrar la cara RÁPIDO)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: No se pudo abrir la cámara.")
    exit()

print("Presiona 'q' para salir...")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Espejar la imagen para que parezca un espejo
    frame = cv2.flip(frame, 1)
    output_frame = frame.copy()
    
    # Usar el detector de OpenCV (rápido) para encontrar la cara
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))

    for (x, labels, w, h) in faces:
        # Recortar el rostro (en BGR, DeepFace prefiere BGR)
        face_img = frame[labels:labels+h, x:x+w]

        # Evitar errores si el recorte está vacío
        if face_img.size == 0:
            continue

        try:
            # 1. Extraer embedding con DeepFace (el mismo modelo de entrenamiento)
            # Usamos 'sQQkip' porque ya le estamos pasando una cara recortada
            embedding_objs = DeepFace.represent(
                img_path=face_img,
                model_name=FACE_MODEL,
                enforce_detection=False, # No es necesario que DeepFace detecte de nuevo
                detector_backend='skip'  # ¡Importante!
            )
            
            # 2. Preparar el embedding para el modelo SVM
            embedding_vector = embedding_objs[0]["embedding"]
            embedding_data = np.array(embedding_vector).reshape(1, -1)

            # 3. Predecir con el modelo SVM
            prediction_idx = model.predict(embedding_data)[0]
            prediction_proba = model.predict_proba(embedding_data)[0]
            
            label = categories[prediction_idx]
            confidence = prediction_proba[prediction_idx] * 100

            # 4. Dibujar resultados
            color = (0, 255, 0) if label == "no_glasses" else (0, 0, 255)
            text = f"{label} ({confidence:.1f}%)"
            
            cv2.rectangle(output_frame, (x, labels), (x+w, labels+h), color, 2)
            cv2.putText(output_frame, text, (x, labels-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        except Exception as e:
            # DeepFace puede fallar si la cara es muy pequeña o está borrosa
            # print(f"Error de DeepFace: Qqqqq{e}") # Descomentar para depurar
            pass

    # Mostrar resultado
    cv2.imshow("Deteccion de Gafas (DeepFace + SVM)", output_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# --- Limpieza ---
cap.release()
cv2.destroyAllWindows()

### Filtro

In [ ]:
import cv2
import numpy as np
import dlib
import joblib
from deepface import DeepFace

def overlay_transparent(background_img, overlay_img, x, y, scale=1.0):
    """
    Superpone una imagen (con canal alfa) sobre otra, con escalado y
    manejo de bordes.
    """
    # Redimensionar la superposición
    overlay_h, overlay_w = overlay_img.shape[:2]
    new_w = int(overlay_w * scale)
    new_h = int(overlay_h * scale)
    
    if new_w == 0 or new_h == 0:
        return background_img
        
    overlay = cv2.resize(overlay_img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    alpha = overlay[:, :, 3] / 255.0
    alpha = alpha[..., np.newaxis]
    rgb = overlay[:, :, :3]

    h, w = background_img.shape[:2]
    y1 = int(y - new_h / 2)
    y2 = int(y + new_h / 2)
    x1 = int(x - new_w / 2)
    x2 = int(x + new_w / 2)

    # --- Manejo de bordes ---
    overlay_y1 = max(0, -y1)
    overlay_y2 = new_h - max(0, y2 - h)
    overlay_x1 = max(0, -x1)
    overlay_x2 = new_w - max(0, x2 - w)

    bg_y1 = max(0, y1)
    bg_y2 = min(h, y2)
    bg_x1 = max(0, x1)
    bg_x2 = min(w, x2)

    if bg_y2 <= bg_y1 or bg_x2 <= bg_x1 or \
       overlay_y2 <= overlay_y1 or overlay_x2 <= overlay_x1:
        return background_img

    roi = background_img[bg_y1:bg_y2, bg_x1:bg_x2]
    alpha_mask = alpha[overlay_y1:overlay_y2, overlay_x1:overlay_x2]
    rgb_overlay = rgb[overlay_y1:overlay_y2, overlay_x1:overlay_x2]

    # --- Fusión ---
    roi[:] = (1.0 - alpha_mask) * roi
    roi[:] = roi + alpha_mask * rgb_overlay
    
    return background_img

def rotate_image(image, angle):
    """
    Gira una imagen (con canal alfa) alrededor de su centro.
    """
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    
    # Obtener matriz de rotación y ajustar el tamaño del bounding box
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos = np.abs(M[0, 0])
    sin = np.abs(M[0, 1])

    # Nuevo bounding box
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))

    # Ajustar la matriz de traslación
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]

    # Aplicar rotación con borde transparente
    return cv2.warpAffine(image, M, (new_w, new_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0,0))

def get_yaw(landmarks):
    """
    Calcula el 'yaw' (giro de cabeza) para el tinte.
    """
    left = landmarks[31]
    right = landmarks[35]
    nose = landmarks[30]
    dx = right[0] - left[0]
    if dx == 0: return 0
    dy = nose[0] - (left[0] + right[0]) / 2
    yaw = dy / dx
    return yaw

# ----------------------
# CONFIG
# ----------------------
MODEL_FILENAME = "glasses_classifier.pkl"
LABELS_FILENAME = "glasses_labels.pkl"
FACE_MODEL = "ArcFace"
GLASSES_SCALE_FACTOR = 1.8 # Ajusta esto para hacer las gafas más grandes o pequeñas

# Cargar clasificador entrenado
clf = joblib.load(MODEL_FILENAME)
class_labels = joblib.load(LABELS_FILENAME)

# Dlib landmarks
try:
    predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
except Exception as e:
    print("Error: No se pudo cargar 'shape_predictor_68_face_landmarks.dat'")

try:
    virtual_glasses_img = cv2.imread("virtual_glasses.png", cv2.IMREAD_UNCHANGED)
    if virtual_glasses_img is None: raise Exception
except Exception as e:
    print("Error: No se pudo cargar 'gafas_virtuales.png'.")
    print("Asegúrate de tener la imagen PNG en la carpeta.")
    virtual_glasses_img = None
# --- FIN DE CAMBIO ---

# Colores para el tinte
TINT_COLORS = [
    (0, 255, 255), (255, 0, 255), (255, 255, 0), (0, 128, 255), (128, 0, 255)
]

# ----------------------
# LOOP PRINCIPAL
# ----------------------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = cv2.flip(frame, 1) # Modo espejo
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    output_frame = frame.copy() # Copia para dibujar

    try:
        # 1. Detección y embedding con DeepFace (flujo corregido)
        embedding_objs = DeepFace.represent(
            img_path=frame,
            model_name=FACE_MODEL,
            enforce_detection=True,
            detector_backend="mtcnn"
        )

        for face in embedding_objs:
            emb = face["embedding"]
            r = face["facial_area"] 
            
            # 2. Predicción SVM
            pred = clf.predict([emb])[0]
            pred_label = class_labels[pred]

            # 3. Obtener Landmarks
            dlib_rect = dlib.rectangle(r['x'], r['y'], r['x'] + r['w'], r['y'] + r['h'])
            shape = predictor(frame_gray, dlib_rect)
            landmarks = np.array([(shape.part(i).x, shape.part(i).y) for i in range(68)])

            # -------------------------------------
            # --- 4. APLICAR FILTRO (LÓGICA 'IF/ELSE') ---
            # -------------------------------------
            
            if pred_label == "glasses":
                # --- SI TIENE GAFAS (Tu código de tinte, está perfecto) ---
                yaw = get_yaw(landmarks)
                yaw_index = int(abs(yaw) * 4) % len(TINT_COLORS)
                tint_color = TINT_COLORS[yaw_index]

                left_eye_pts = landmarks[36:42]
                right_eye_pts = landmarks[42:48]

                def tint_region(pts):
                    mask = np.zeros(output_frame.shape, dtype=np.uint8)
                    cv2.fillConvexPoly(mask, pts, tint_color)
                    return cv2.addWeighted(output_frame, 1, mask, 0.4, 0)

                output_frame = tint_region(left_eye_pts)
                output_frame = tint_region(right_eye_pts)

                cv2.putText(output_frame, "GLASSES DETECTED", (r['x'], r['y'] - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            else:
                if virtual_glasses_img is not None:
                
                    # 1. Calcular pose de las gafas
                    left_eye_corner = landmarks[36]
                    right_eye_corner = landmarks[45]
                    
                    # Calcular centro (entre los puntos 39 y 42)
                    center_x = int((landmarks[39][0] + landmarks[42][0]) / 2)
                    center_y = int((landmarks[39][1] + landmarks[42][1]) / 2)

                    # Calcular ancho y ángulo
                    glasses_width = np.linalg.norm(right_eye_corner - left_eye_corner)
                    angle_deg = np.degrees(np.arctan2(
                        right_eye_corner[1] - left_eye_corner[1],
                        right_eye_corner[0] - left_eye_corner[0]
                    ))

                    # 2. Rotar la imagen de las gafas
                    rotated_glasses = rotate_image(virtual_glasses_img, angle_deg)
                    
                    # 3. Calcular escala
                    scale = (glasses_width / virtual_glasses_img.shape[1]) * GLASSES_SCALE_FACTOR
                    
                    # 4. Superponer
                    output_frame = overlay_transparent(output_frame, rotated_glasses, center_x, center_y, scale)

                cv2.putText(output_frame, "COGE UNAS GAFAS, COLEGA!", (r['x'], r['y'] - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    except ValueError as e:
        # (DeepFace no encuentra cara)
        pass
    except Exception as e:
        # (Otro error, ej. dlib)
        # print(e) # Descomentar para depurar
        pass

    cv2.imshow("Filtro gafas (Propuesta 1)", output_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()